# Cat Embeddings Model
ShuffleNetV2 x0.5 repurposed to create a lightweight embeddings model for cat identity and behaviour classification

## Configure notebook

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import silhouette_score
from torchvision import models

from config import settings

In [ ]:
%matplotlib inline

In [ ]:
# load dataset manifest
data_path = Path("datasets") / "classification_data"
manifest_df = pd.read_csv(data_path / "labels_manifest.csv").astype(
    {"cat_name": "category"}
)
manifest_df["image_path"] = manifest_df.apply(
    lambda row: data_path / "images" / row["split"] / row["image_name"], axis=1
)

## Build embedding model dictionary

In [ ]:
def build_embedding_backbone(model_name, weights):
    model = getattr(models, model_name)(weights=weights).eval()
    if model_name.startswith("shufflenet_v2"):
        embedder = torch.nn.Sequential(
            model.conv1,
            model.maxpool,
            model.stage2,
            model.stage3,
            model.stage4,
            model.conv5,
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(1),
        )
    elif model_name.startswith("mobilenet_v3"):
        embedder = torch.nn.Sequential(
            model.features,
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(1),
        )
    elif model_name.startswith("efficientnet_b"):
        embedder = torch.nn.Sequential(
            model.features,
            model.avgpool,
            torch.nn.Flatten(1),
        )
    else:
        raise ValueError(f"Unsupported model architecture: {model_name}")
    return embedder


model_specs = {
    "ShuffleNetV2 0.5x": {
        "model_name": "shufflenet_v2_x0_5",
        "weights": models.ShuffleNet_V2_X0_5_Weights.DEFAULT,
    },
    "ShuffleNetV2 1.0x": {
        "model_name": "shufflenet_v2_x1_0",
        "weights": models.ShuffleNet_V2_X1_0_Weights.DEFAULT,
    },
    "MobileNetV3-Small": {
        "model_name": "mobilenet_v3_small",
        "weights": models.MobileNet_V3_Small_Weights.DEFAULT,
    },
    "MobileNetV3-Large": {
        "model_name": "mobilenet_v3_large",
        "weights": models.MobileNet_V3_Large_Weights.DEFAULT,
    },
    "EfficientNet-B0": {
        "model_name": "efficientnet_b0",
        "weights": models.EfficientNet_B0_Weights.DEFAULT,
    },
    "EfficientNet-B1": {
        "model_name": "efficientnet_b1",
        "weights": models.EfficientNet_B1_Weights.DEFAULT,
    },
}

model_options = {}
for model_label, spec in model_specs.items():
    preprocess = spec["weights"].transforms(
        crop_size=settings.EMBEDDING_IMGSZ, resize_size=settings.EMBEDDING_IMGSZ
    )
    embedding_model = build_embedding_backbone(spec["model_name"], spec["weights"])
    model_options[model_label] = {
        "embedding_model": embedding_model.eval(),
        "preprocess": preprocess,
    }

## Evaluate models with inference time and silhouette score

In [ ]:
results = []
embeddings_by_model = {}

for model_label, model_bundle in model_options.items():
    embedding_model = model_bundle["embedding_model"]
    preprocess = model_bundle["preprocess"]

    # warm-up pass
    warmup_image = Image.open(manifest_df["image_path"].iloc[0]).convert("RGB")
    warmup_batch = preprocess(warmup_image).unsqueeze(0)
    with torch.inference_mode():
        _ = embedding_model(warmup_batch)

    # calculate and time embeddings
    timings_ms = []
    vectors = []
    for image_path in manifest_df["image_path"]:
        batch = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0)
        start = time.perf_counter()
        with torch.inference_mode():
            embedding = embedding_model(batch)
        timings_ms.append((time.perf_counter() - start))
        vectors.append(embedding.squeeze(0).detach().cpu().numpy())
    embeddings = np.stack(vectors)
    embeddings_by_model[model_label] = {
        "embeddings": embeddings,
        "class_labels": manifest_df["cat_name"],
    }

    # store results
    results.append(
        {
            "model": model_label,
            "mean_inference_ms": np.mean(timings_ms) * 1000,
            "silhouette": silhouette_score(
                embeddings, manifest_df["cat_name"].cat.codes
            ),
        }
    )

results_df = pd.DataFrame(results)
results_df.sort_values(["silhouette", "mean_inference_ms"], ascending=[False, True])

In [ ]:
# display model comparison
ax = results_df.plot.scatter(
    x="mean_inference_ms", y="silhouette", s=150, figsize=(15, 10)
)
_ = [
    ax.annotate(m, (x, y), xytext=(-10, 10), textcoords="offset points")
    for m, x, y in zip(
        results_df["model"], results_df["mean_inference_ms"], results_df["silhouette"]
    )
]

## Export best model to ONNX

In [ ]:
# select model
best_model_label = "ShuffleNetV2 0.5x"
best_model_spec = model_specs[best_model_label]
best_embedding_model = model_options[best_model_label]["embedding_model"]

# prepare model for export
onnx_path = Path("models_staging") / f"{settings.MODEL_EMBEDDING_PATH}.onnx"
onnx_path.parent.mkdir(parents=True, exist_ok=True)
quantized_model = best_embedding_model.to("cpu").half().eval()

# export to ONNX
dummy = torch.randn(
    1, 3, settings.EMBEDDING_IMGSZ, settings.EMBEDDING_IMGSZ, dtype=torch.float16
)
onnx_program = torch.onnx.export(
    quantized_model,
    (dummy,),
    input_names=["images"],
    output_names=["embeddings"],
    opset_version=18,
    dynamic_shapes=None,
)
onnx_program.save(onnx_path, external_data=False)